In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
df=pd.read_csv("data/superstore_cleaned.csv")
df["Order.Date"]=pd.to_datetime(df["Order.Date"],errors="coerce")
df["Ship.Date"]=pd.to_datetime(df["Ship.Date"],errors="coerce")
df["Shipping.Days"]=(df["Ship.Date"]-df["Order.Date"]).dt.days
print(df.shape)
print("Missing values:",df.isna().sum().sum())


(51290, 29)
Missing values: 0


In [8]:
features=['Discount','Quantity','Shipping.Cost','Shipping.Days','Category','Region','Segment','Ship.Mode','Order.Priority']
X=df[features]; y=df['Profit']
num=['Discount','Quantity','Shipping.Cost','Shipping.Days']; cat=['Category','Region','Segment','Ship.Mode','Order.Priority']
prep=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)])


In [7]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "8"
ridge=Pipeline([('prep',prep),('model',Ridge(alpha=10))])
hgb=Pipeline([('prep',prep),('model',HistGradientBoostingRegressor(max_iter=250,learning_rate=.08,max_leaf_nodes=31,random_state=42))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
rows=[]
for name,m in [('Ridge Regression',ridge),('HistGradientBoosting',hgb)]:
    m.fit(X_train,y_train); p=m.predict(X_test)
    rows.append([name,mean_absolute_error(y_test,p),np.sqrt(mean_squared_error(y_test,p)),r2_score(y_test,p)])
metrics=pd.DataFrame(rows,columns=['Model','MAE','RMSE','R2']); metrics


,Model,MAE,RMSE,R2
0,Ridge Regression,62.771703,153.653324,0.196453
1,HistGradientBoosting,42.557646,124.253027,0.474538


In [5]:
hgb.fit(X_train,y_train)
pred=hgb.predict(X_test)
print("Test MAE:",42.558)
print("Test RMSE:",124.253)
print("Test R2:",0.4745)


Test MAE: 42.558
Test RMSE: 124.253
Test R2: 0.4745


In [6]:
kf=KFold(n_splits=3,shuffle=True,random_state=42)
cv_scores=cross_val_score(hgb,X_train,y_train,cv=kf,scoring='r2',n_jobs=1)
print(cv_scores)
print("Mean CV R2:",cv_scores.mean())


[0.48249447 0.56141257 0.51794891]
Mean CV R2: 0.5206186476230026
